# RE_205 · Quantes bicis caldran a les 8?

**Optativa d'Aprenentatge automàtic · Bloc 2 — Supervisat · Dificultat 2 · durada: 1 o 2 sessions de 2 h**

Treballes per a un servei de bicicleta compartida, com el Bicing. Cada nit, l'equip de logística
redistribueix les bicis entre estacions, i per fer-ho bé necessita saber **quantes se'n llogaran cada
hora del dia següent**. Tens dos anys d'històric amb el temps que feia i el tipus de dia.

**La pregunta:** quin model preveu millor els lloguers per hora, i com l'has d'avaluar perquè l'error que
prometis sigui el que hi haurà l'any que ve?

## Les dades

- **Font:** *Bike Sharing* (Fanaee-T i Gama, 2013; UCI), a OpenML amb `data_id=42712`. Llicència
  CC BY 4.0. Dades del sistema Capital Bikeshare de **Washington DC**, 2011 i 2012.
- **Cada fila** és **una hora**: 17.379 files i 13 columnes.
- Documentació: <https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset>

### Diccionari de columnes

| Columna | Què és |
| --- | --- |
| `count` | **objectiu**: bicis llogades en aquella hora |
| `season` | estació de l'any: `spring`, `summer`, `fall`, `winter` |
| `year` | any: 0 = 2011, 1 = 2012 |
| `month` | mes (1-12) |
| `hour` | hora del dia (0-23) |
| `holiday` | si és festiu |
| `weekday` | dia de la setmana (0 = diumenge ... 6 = dissabte) |
| `workingday` | si és laborable (ni cap de setmana ni festiu) |
| `weather` | temps: `clear`, `misty` (boira o núvols), `rain` (pluja o neu lleugeres), `heavy_rain` (pluja forta, tempesta) |
| `temp`, `feel_temp` | temperatura real i sensació tèrmica, en °C |
| `humidity` | humitat relativa, de 0 a 1 |
| `windspeed` | velocitat del vent |

### Com s'han recollit, i què cal tenir present

- El sistema registra cada lloguer; aquí estan sumats per hora i creuats amb el temps de l'hora i el
  calendari de festius de Washington.
- Una hora sense cap lloguer no té fila.

## Què has d'investigar

Prepara la previsió per a logística. Mira primer la forma del dia: com són els lloguers per hora
en un laborable i en un festiu, i què et diu sobre qui fa servir les bicis. Parteix les dades
**pel temps** (entrenament 2011, test 2012), perquè el model s'ha de fer servir l'any que ve, i
compara models: una **recta** amb l'hora com a número, la recta amb l'hora en one-hot, i un
**bosc**. Quin aprofita millor l'hora, i per què?

La teoria ha d'aterrar al codi: comprova que `predict` de la recta és exactament $Xw + b$
calculant-ho tu amb la matriu transformada; implementa el MAE i l'RMSE i comprova'ls amb
scikit-learn. Després, entrena el mateix bosc amb una partició **aleatòria** i compara l'error amb
el de la temporal. Quin número posaries a l'informe? Acaba mirant on s'equivoca el model: l'error
té signe? Varia per mesos?

**Per anar més enllà** (no és obligatori, i no hi ha una sola manera de fer-ho):

- Escriu l'hora com a dues columnes, $\sin(2\pi h/24)$ i $\cos(2\pi h/24)$. Millora la recta? Per què no arriba al one-hot?
- Una columna que combini laborable i hora (per exemple, `"1_8"`): quant s'acosta la recta al bosc?

**Pistes per pensar** (no són la resposta):

- Si una partició et dona la meitat d'error que l'altra, pregunta't quina de les dues s'assembla a com es farà servir el model.
- Un error que té sempre el mateix signe no és soroll: és una cosa que el model no sap.

## Exemples per inspirar-te

Quaderns del curs on es fan servir les eines que et caldran. Mira com es treballa, no hi busquis
la resposta: les dades d'aquest repte són unes altres.

- [MA_01 · àlgebra lineal](../03_matematiques/MA_01_algebra_lineal.ipynb) §7 (un model lineal és un vector i una multiplicació) i [MA_02 · descens del gradient](../03_matematiques/MA_02_descens_gradient.ipynb) §3-§4 (minimitzar un error).
- [ML_02 · arbres](../01_teoria/ML_02_arbres.ipynb) i [ML_03 · boscos](../01_teoria/ML_03_boscos.ipynb): arbres i boscos.
- [EX_00 · caixa d'eines](../02_practica/EX_00_caixa_eines.ipynb) §1 (`train_test_split` i el seu parany), §4 i §6 (`OneHotEncoder` i `ColumnTransformer`).
- MAE i $R^2$: si no has fet el RE_204, mira'n l'enunciat.

## Lliurament

Lliura aquest quadern **executat**, sense errors, amb la teva anàlisi: cel·les de codi i, entre elles, cel·les de text on expliques què has fet i per què. Cada decisió que prenguis (què fas amb un buit, quina columna treus, quin model tries) ha de tenir la seva línia de justificació. Acaba amb una **conclusió** de tres a cinc frases, escrita per tu, que respongui l'encàrrec amb els números que has trobat.

> **Quan cridar en Xavi:** per a dubtes de **criteri**, no de sintaxi (per a la sintaxi hi ha la
> documentació i els exemples). Per exemple: «La partició aleatòria em dona molt millor resultat. Per què no la faig servir?» «El model es queda curt sempre. Ho puc corregir, o és trampa?»

La cel·la següent carrega les dades. A partir d'aquí el quadern és teu: afegeix-hi totes les
cel·les de codi i de text que necessitis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=42712, as_frame=True).frame
print(df.shape)
df.head()